In [ ]:
import gc
import pickle
import argparse
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.autograd import Variable
import torch.nn.functional as F

# 老师服务器内存不够，这个实在跑不完ORZ😢

In [ ]:
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    tot = pickle.load(f)

In [ ]:
import os

# 使用 free -h 命令查看内存使用情况
memory_info = os.popen('free -h').read()
print(memory_info)

In [ ]:
parser = argparse.ArgumentParser()

# TODO 常改动参数
parser.add_argument('--gpu', default=0, type=int) # gpu 卡号
parser.add_argument('--epochs', default=100, type=int) # 训练轮数
parser.add_argument('--layers', default=2, type=int) # LSTM层数
parser.add_argument('--input_size', default=920, type=int) #输入特征的维度
parser.add_argument('--hidden_size', default=256, type=int) #隐藏层的维度
parser.add_argument('--lr', default=0.01, type=float) #learning rate 学习率
parser.add_argument('--sequence_length', default=100, type=int) # sequence的长度
parser.add_argument('--batch_size', default=1024, type=int)
parser.add_argument('--useGPU', default=False, type=bool) #是否使用GPU
parser.add_argument('--batch_first', default=True, type=bool) #是否将batch_size放在第一维
parser.add_argument('--dropout', default=0.1, type=float)
parser.add_argument('--save_file', default='model/stock.pkl') # 模型保存位置

args, unknown = parser.parse_known_args()

device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() and args.useGPU else "cpu")
args.device = device

In [ ]:
class lstm(nn.Module):
    def __init__(self, input_size=8, hidden_size=32, num_layers=1 , output_size=1 , dropout=0, batch_first=True):
        super(lstm, self).__init__()
        # lstm的输入 #batch,seq_len, input_size
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers
        self.output_size = output_size
        self.dropout = dropout
        self.batch_first = batch_first
        self.rnn = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size, num_layers=self.num_layers, batch_first=self.batch_first, dropout=self.dropout )
        self.linear = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, x):
        out, (hidden, cell) = self.rnn(x)  # x.shape : batch, seq_len, hidden_size , hn.shape and cn.shape : num_layes * direction_numbers, batch, hidden_size
        # a, b, c = hidden.shape
        # out = self.linear(hidden.reshape(a * b, c))
        out = self.linear(hidden)
        return out
def create_sequences(datas, seq_length):
    X = []
    Y = []
    XX = datas.drop(['permno', 'DATE', 'RET'], axis=1).values
    yy = datas['RET'].values
    for i in range(len(datas) - seq_length):
        X.append(XX[i:(i + seq_length), :])
        Y.append(yy[i + seq_length])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

In [ ]:
tot['DATE'] = pd.to_datetime(tot['DATE'], format='%Y%m')

In [ ]:
tot.groupby('DATE').size().mean()

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
tot['DATE'] = pd.to_datetime(tot['DATE'], format='%Y%m')

In [ ]:
model = lstm(input_size=args.input_size, hidden_size=args.hidden_size, num_layers=args.layers , output_size=1, dropout=args.dropout, batch_first=args.batch_first )
model.to(args.device)
criterion = nn.MSELoss()  # 定义损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)  # Adam梯度下降  学习率=0.001
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year-1, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    # train_data = data0[(data0['DATE'] >= start_date_train) & (data0['DATE'] <= end_date_val)]
    # test_data = data0[(data0['DATE'] >= start_date_test) & (data0['DATE'] <= end_date_test)]

    train_data = tot[(tot['DATE'] >= start_date_train) & (tot['DATE'] <= end_date_val)]
    test_data = tot[(tot['DATE'] >= start_date_test) & (tot['DATE'] <= end_date_test)]
    
    X_test = torch.tensor(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_data['RET'].values, dtype=torch.float32).view(-1, 1)
    
    sequence = args.sequence_length

    # 构建训练和测试的时间序列
    train_sequences, train_labels = create_sequences(train_data, sequence)
    test_sequences, test_labels = create_sequences(test_data, sequence)
    
    train_dataset = TensorDataset(train_sequences, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=False)
    test_dataset = TensorDataset(test_sequences, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # for batch in test_loader:
    #     print(f"Batch X shape: {batch[0].shape}")
    #     print(f"Batch y shape: {batch[1].shape}")
    #     break
    
    patience = 20  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(args.epochs), desc='Epoch', unit='item'):
        model.train()
        train_loss = 0
        for idx, (data, label) in enumerate(train_loader):
            if args.useGPU:
                data1 = data.squeeze(1).cuda()
                pred = model(Variable(data1).cuda())
                # print(pred.shape)
                pred = pred[1,:,:]
                label = label.unsqueeze(1).cuda()
                # print(label.shape)
            else:
                data1 = data.squeeze(1)
                # print(f'train_size{data.shape}')
                pred = model(Variable(data1))
                pred = pred[1, :, :]
                label = label.unsqueeze(1)
                # print(f'label_size{label.shape}')
                # print(f'pred_size{pred.shape}')
            train_r2 = r2_score(label.detach(), pred.detach())
            loss = criterion(pred, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        # print(f'train_loss:{train_loss}')
        # print(f'train_r2:{train_r2}')
        model.eval()  # 设置模型为评估模式
        test_loss=0
        preds=[]
        labels=[]
        # print(f'test_data_size{X_test.unsqueeze(0).shape}')
        # print(f'test_label_size{y_test.shape}')
        # pred= model(Variable(X_test.unsqueeze(0)))
        # pred=pred[1, :, :]
        # print(pred.shape)
        # print(y_test.shape)
        for idx, (data, label) in enumerate(test_loader):
            if args.useGPU:
                data = data.squeeze(1).cuda()  # batch_size,seq_len,input_size
            else:
                data = data.squeeze(1)
                pred = model(data)[1,:,:].item()
                preds.append(pred)
                labels.append(label.item())
        preds=torch.tensor(preds, dtype=torch.float32)
        labels=torch.tensor(labels, dtype=torch.float32)
        loss = criterion(preds, labels)
        test_loss = loss.item()
        test_r2 = r2_score(labels, preds)
        # print(f'test_loss:{test_loss}')
        # print(f'test_r2:{test_r2}')
        print(f'Epoch [{epoch + 1}/{args.epochs}], Train : Loss: {train_loss:.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss:.4f}, R²: {test_r2:.4f}')
     
#         # 每 10 个 epoch 打印一次损失和 R² 分数
        if test_loss<best_loss:
            best_loss = test_loss
            patience_counter = 0
            # 更新最佳模型状态
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch + 1}')
                break
    # Predict on test data
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.eval()  # 设置模型为评估模式
    preds=[]
    labels=[]
    for idx, (data, label) in enumerate(test_loader):
        if args.useGPU:
            data = data.squeeze(1).cuda()  # batch_size,seq_len,input_size
        else:
            data = data.squeeze(1)
            pred = model(data)[1,:,:].item()
            preds.append(pred)
            labels.append(label.item())
    pred_y_list.extend(preds)
    true_y_list.extend(labels)
# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(true_y_list,pred_y_list)))
gc.collect()